# 14 · Transformer Architecture and Self-Attention

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anuxlab/deep-learning-systems/blob/main/notebooks/14-transformer/14_transformer.ipynb)
[![Run on GitHub Actions](https://img.shields.io/badge/run-GitHub%20Actions-2088FF?logo=githubactions&logoColor=white)](https://github.com/anuxlab/deep-learning-systems/actions/workflows/nb-14-transformer.yml)

| | |
|---|---|
| **Wiki chapter** | [Module 14](https://github.com/anuxlab/deep-learning-systems/wiki/Module-14) |
| **Level** | Advanced |
| **Typical CPU runtime** | ~2 min (GitHub free runner, no GPU) |
| **Prerequisites** | Modules 09, 13 |

## What you will do

- Derive and implement scaled dot-product attention and verify a hand-computed example.
- Show why $1/\sqrt{d_k}$ is needed, and test causal/padding masks.
- Show permutation equivariance of self-attention without positional information.
- Implement multi-head attention, the FFN and pre-LN blocks; match `nn.MultiheadAttention`.
- Count parameters and attention cost, and train a tiny decoder-only Transformer.

> **How to use this notebook.** Read each markdown cell, *predict* what the next code cell will print, then run it.
> Every section ends with `check(...)` assertions: if one fails, something in your understanding (or the code) is off.
> Figures are saved to `figures/` and a machine-readable summary to `metrics.json` next to this notebook.

## 0 · Setup and reproducibility

We import shared helpers (`show` saves each figure to `figures/`, `check` asserts and logs, `timer` logs wall-clock time), fix all random seeds and print an environment report so every run is traceable (see the wiki's *Implementation and Research Standards*).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "_common"))   # shared helpers: notebooks/_common/dlsys.py
from dlsys import *                                       # np, plt, torch, seed_everything, show, check, timer, ...

SEED = seed_everything(42)
ENV = env_report()

## 1 · Scaled dot-product attention
$$\mathrm{Attn}(Q,K,V)=\mathrm{softmax}\!\Big(\frac{QK^\top}{\sqrt{d_k}}\Big)V,\qquad Q\in\mathbb R^{T\times d_k},K\in\mathbb R^{S\times d_k},V\in\mathbb R^{S\times d_v}.$$
Hand-checkable example: $T=S=2$, $d_k=d_v=2$.

In [ ]:
def attention(Q, K, V, mask=None):
    d = Q.shape[-1]; S_ = Q @ K.swapaxes(-1, -2) / np.sqrt(d)
    if mask is not None: S_ = np.where(mask, S_, -1e9)
    P = np.exp(S_ - S_.max(-1, keepdims=True)); P /= P.sum(-1, keepdims=True); return P @ V, P
Q = np.array([[1., 0.], [0., 1.]]); K = np.array([[1., 0.], [1., 1.]]); V = np.array([[1., 2.], [3., 4.]])
out, P = attention(Q, K, V); print("scores/√2 =\n", (Q @ K.T / np.sqrt(2)).round(4)); print("P =\n", P.round(4)); print("output =\n", out.round(4))
exp_row0 = np.exp(np.array([1, 1]) / np.sqrt(2)); exp_row0 /= exp_row0.sum()
check(np.allclose(P[0], exp_row0) and np.allclose(P[0], [.5, .5]), "row 0: scores (1,1)/√2 are equal -> weights (.5,.5)")
check(np.allclose(out[0], [2, 3]), "row 0 output = average of the two value vectors (2,3)")
check(np.allclose(P.sum(-1), 1), "attention weights are row-stochastic")
check(np.allclose(out, F.scaled_dot_product_attention(torch.tensor(Q)[None], torch.tensor(K)[None], torch.tensor(V)[None])[0].numpy()), "matches torch.nn.functional.scaled_dot_product_attention")

### Why divide by $\sqrt{d_k}$?
If entries of $q,k$ are independent with zero mean and unit variance, $q^\top k$ has variance $d_k$. Large-magnitude scores push softmax into saturation (almost one-hot) where gradients vanish. Dividing by $\sqrt{d_k}$ restores unit variance. We verify the variance and the saturation empirically.

In [ ]:
rng = np.random.default_rng(0)
for d in [4, 64, 1024]:
    q, k = rng.normal(size=(20000, d)), rng.normal(size=(20000, d)); sc = (q * k).sum(1)
    print(f"d_k={d:5d}: var(q·k) = {sc.var():9.1f} (theory {d})  var(q·k/√d) = {(sc/np.sqrt(d)).var():.2f}")
    check(abs(sc.var() / d - 1) < .08 and abs((sc / np.sqrt(d)).var() - 1) < .08, f"d_k={d}: variance d -> 1 after scaling")
def max_weight(d, scale): q = rng.normal(size=d); K_ = rng.normal(size=(32, d)); e = K_ @ q * scale; p = np.exp(e - e.max()); p /= p.sum(); return p.max()
mw = {d: (np.mean([max_weight(d, 1) for _ in range(200)]), np.mean([max_weight(d, 1 / np.sqrt(d)) for _ in range(200)])) for d in [4, 64, 1024]}
print("mean max attention weight (unscaled, scaled):", {d: tuple(np.round(v, 2)) for d, v in mw.items()})
check(mw[1024][0] > .95 and mw[1024][1] < .5, "d_k=1024: unscaled softmax is near one-hot; scaled stays soft")

## 2 · Masks: causal and padding
- **Causal mask** (decoder / language model): position $i$ may attend only to $j\le i$ — a lower-triangular boolean matrix; future scores are set to $-\infty$ before the softmax so their weights are exactly 0.
- **Padding mask**: sequences in a batch are padded to equal length; padded keys must get zero weight.

The key test of a causal mask: **changing future tokens must not change past outputs**.

In [ ]:
T, d = 6, 8; X = rng.normal(size=(T, d)); causal = np.tril(np.ones((T, T), bool))
Wq, Wk, Wv = [rng.normal(size=(d, d)) / np.sqrt(d) for _ in range(3)]
def self_attn(X, mask=None): return attention(X @ Wq, X @ Wk, X @ Wv, mask)
o1, P1 = self_attn(X, causal); X2 = X.copy(); X2[4:] = rng.normal(size=(2, d)); o2, _ = self_attn(X2, causal)
check(np.allclose(P1[np.triu_indices(T, 1)], 0), "causal weights above the diagonal are exactly 0")
check(np.allclose(o1[:4], o2[:4]) and not np.allclose(o1[4:], o2[4:]), "editing future tokens leaves earlier outputs unchanged")
pad = np.array([True] * 4 + [False] * 2); o3, P3 = self_attn(X, np.broadcast_to(pad[None], (T, T)))
check(np.allclose(P3[:, 4:], 0), "padding mask: padded keys get zero weight")
fig, ax = plt.subplots(1, 2, figsize=(9, 3.5)); ax[0].imshow(P1, cmap="viridis"); ax[0].set_title("causal attention weights"); ax[1].imshow(P3, cmap="viridis"); ax[1].set_title("padding-masked (last 2 keys padded)"); show("attention_masks")

## 3 · Permutation equivariance — why positions are needed
Pure self-attention treats its inputs as a **set**: permuting the input tokens simply permutes the outputs, $f(\Pi X)=\Pi f(X)$. It cannot tell "dog bites man" from "man bites dog" unless position information is injected (Module 15). We verify equivariance and then show that a causal mask *breaks* it (because the mask itself encodes order).

In [ ]:
perm = rng.permutation(T); o_a, _ = self_attn(X); o_b, _ = self_attn(X[perm])
check(np.allclose(o_a[perm], o_b), "self-attention is permutation-equivariant: f(ΠX) = Πf(X)")
oc_a, _ = self_attn(X, causal); oc_b, _ = self_attn(X[perm], causal)
check(not np.allclose(oc_a[perm], oc_b), "with a causal mask equivariance is broken (order matters)")

## 4 · Multi-head attention, FFN, pre-LN blocks (PyTorch)
$h$ heads each of width $d_k=d/h$: project, split, attend in parallel, concatenate, project with $W_O$. Different heads can specialise in different relations at the same cost as one big head. We implement it with explicit tensor shapes and **validate against `nn.MultiheadAttention`** by copying weights.

In [ ]:
class MHA(nn.Module):
    def __init__(s, d, h): super().__init__(); s.h, s.dk = h, d // h; s.qkv = nn.Linear(d, 3 * d); s.o = nn.Linear(d, d)
    def forward(s, x, causal=False, pad_mask=None, return_attn=False):                      # x: (B,T,d)
        B, T, d = x.shape; q, k, v = s.qkv(x).view(B, T, 3, s.h, s.dk).permute(2, 0, 3, 1, 4)       # each (B,h,T,dk)
        sc = q @ k.transpose(-1, -2) / s.dk ** .5                                                  # (B,h,T,T)
        if causal: sc = sc.masked_fill(~torch.tril(torch.ones(T, T, dtype=torch.bool)), float("-inf"))
        if pad_mask is not None: sc = sc.masked_fill(~pad_mask[:, None, None, :], float("-inf"))
        a = F.softmax(sc, -1); y = (a @ v).transpose(1, 2).reshape(B, T, d); y = s.o(y); return (y, a) if return_attn else y
torch.manual_seed(0); d, h = 32, 4; mine = MHA(d, h); ref = nn.MultiheadAttention(d, h, batch_first=True)
with torch.no_grad(): ref.in_proj_weight.copy_(mine.qkv.weight); ref.in_proj_bias.copy_(mine.qkv.bias); ref.out_proj.weight.copy_(mine.o.weight); ref.out_proj.bias.copy_(mine.o.bias)
x = torch.randn(3, 10, d); y1 = mine(x); y2, _ = ref(x, x, x, need_weights=False)
check(torch.allclose(y1, y2, atol=1e-5), "our MHA == nn.MultiheadAttention (no mask)")
cm = torch.triu(torch.ones(10, 10, dtype=torch.bool), 1); y3 = mine(x, causal=True); y4, _ = ref(x, x, x, attn_mask=cm, need_weights=False)
check(torch.allclose(y3, y4, atol=1e-5), "our MHA == nn.MultiheadAttention (causal)")
print("shapes: qkv", tuple(mine.qkv(x).shape), "-> heads (B,h,T,dk) =", (3, h, 10, d // h), "-> out", tuple(y1.shape))

In [ ]:
class Block(nn.Module):                                   # pre-LN Transformer block
    def __init__(s, d, h, ff=4, p=0.0): super().__init__(); s.n1, s.n2 = nn.LayerNorm(d), nn.LayerNorm(d); s.att = MHA(d, h); s.ffn = nn.Sequential(nn.Linear(d, ff * d), nn.GELU(), nn.Linear(ff * d, d)); s.drop = nn.Dropout(p)
    def forward(s, x, **kw): x = x + s.drop(s.att(s.n1(x), **kw)); return x + s.drop(s.ffn(s.n2(x)))
blk = Block(64, 8); xb = torch.randn(2, 12, 64); check(blk(xb, causal=True).shape == xb.shape, "block preserves shape (B,T,d)")
d, ff = 64, 4; expected = (3 * d * d + 3 * d) + (d * d + d) + (d * ff * d + ff * d) + (ff * d * d + d) + 2 * (2 * d)
print("parameters in one block:", count_params(blk), " formula 12d²+13d =", 12 * d * d + 13 * d)
check(count_params(blk) == expected == 12 * d * d + 13 * d, "block parameters = 12d² + 13d (attention 4d², FFN 8d²)")

## 5 · Attention cost: $O(T^2)$
Computing the $T\times T$ score matrix costs $O(T^2d)$ FLOPs and $O(T^2)$ memory per head. Doubling the context quadruples the attention cost, while the FFN part grows only linearly. We measure wall-clock time.

In [ ]:
import time
Ts = [128, 256, 512, 1024, 2048]; times = []
att = MHA(64, 4).eval()
with torch.no_grad():
    for T_ in Ts:
        x = torch.randn(1, T_, 64); att(x); t0 = time.perf_counter()
        for _ in range(3): att(x)
        times.append((time.perf_counter() - t0) / 3)
for a_, b_ in zip(Ts, times): print(f"T={a_:5d}: {b_*1e3:8.1f} ms")
slope = np.polyfit(np.log(Ts[2:]), np.log(times[2:]), 1)[0]; print("empirical scaling exponent (large T):", round(slope, 2))
plt.loglog(Ts, times, "o-", label="measured"); plt.loglog(Ts, times[0] * (np.array(Ts) / Ts[0]) ** 2, "k--", label="∝ T²"); plt.xlabel("sequence length T"); plt.ylabel("seconds"); plt.legend(); plt.title("Self-attention cost"); show("attention_cost")
check(slope > 1.3, "attention time grows super-linearly with T"); METRICS["attention_scaling_exponent"] = slope

## 6 · A tiny decoder-only Transformer (GPT-style)
Token + learned position embeddings → $N$ pre-LN blocks with causal attention → LayerNorm → tied output projection. Trained on the *Zen of Python* as a character LM (offline). We check that it (i) overfits the tiny corpus well below the uniform-baseline loss, (ii) obeys causality at the model level, and (iii) inspect an attention pattern.

In [ ]:
import codecs, this
text = codecs.decode(this.s, "rot13"); chars = sorted(set(text)); stoi = {c: i for i, c in enumerate(chars)}; itos = dict(enumerate(chars)); data = torch.tensor([stoi[c] for c in text]); V = len(chars)
class GPT(nn.Module):
    def __init__(s, V, d=96, h=4, n=3, ctx=64):
        super().__init__(); s.tok, s.pos = nn.Embedding(V, d), nn.Embedding(ctx, d); s.blocks = nn.ModuleList([Block(d, h) for _ in range(n)]); s.nf = nn.LayerNorm(d); s.head = nn.Linear(d, V, bias=False); s.head.weight = s.tok.weight; s.ctx = ctx
    def forward(s, idx):
        x = s.tok(idx) + s.pos(torch.arange(idx.shape[1])); 
        for b in s.blocks: x = b(x, causal=True)
        return s.head(s.nf(x))
torch.manual_seed(0); gpt = GPT(V); o = torch.optim.AdamW(gpt.parameters(), 2e-3, weight_decay=.01); print("GPT parameters:", f"{count_params(gpt):,}"); L = 64; losses = []
with timer("gpt_training"):
    for step in range(500):
        ix = torch.randint(0, len(data) - L - 1, (32,)); x = torch.stack([data[i:i + L] for i in ix]); y = torch.stack([data[i + 1:i + L + 1] for i in ix])
        loss = F.cross_entropy(gpt(x).reshape(-1, V), y.reshape(-1)); o.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(gpt.parameters(), 1.0); o.step(); losses.append(loss.item())
        if step % 100 == 0: print(f"step {step:4d} loss {loss.item():.3f}")
print(f"final loss {np.mean(losses[-20:]):.3f} vs uniform {np.log(V):.3f}"); plt.plot(losses); plt.xlabel("step"); plt.ylabel("loss"); plt.title("Tiny GPT training loss"); show("gpt_loss")
check(np.mean(losses[-20:]) < .5 * np.log(V), "GPT loss is below half the uniform baseline"); METRICS["gpt_final_loss"] = float(np.mean(losses[-20:]))
gpt.eval(); xi = data[:32][None].clone(); l1 = gpt(xi); xi2 = xi.clone(); xi2[0, 20:] = torch.randint(0, V, (12,)); l2 = gpt(xi2)
check(torch.allclose(l1[0, :20], l2[0, :20], atol=1e-5), "model-level causality: logits at positions <20 ignore edits at positions ≥20")
@torch.no_grad()
def generate(m, start, n=150, temp=.7):
    idx = torch.tensor([[stoi[c] for c in start]])
    for _ in range(n): lg = m(idx[:, -m.ctx:])[0, -1] / temp; idx = torch.cat([idx, torch.multinomial(F.softmax(lg, -1), 1)[None]], 1)
    return "".join(itos[i] for i in idx[0].tolist())
print(generate(gpt, "Errors should")); 
with torch.no_grad(): _, A = gpt.blocks[0].att(gpt.blocks[0].n1(gpt.tok(xi[:, :24]) + gpt.pos(torch.arange(24))), causal=True, return_attn=True)
fig, ax = plt.subplots(1, 4, figsize=(14, 3.2))
for hh in range(4): ax[hh].imshow(A[0, hh], cmap="viridis"); ax[hh].set_title(f"layer 0, head {hh}")
show("gpt_attention_heads")

In [ ]:
save_metrics("14", METRICS)
log.info('notebook 14 finished OK')

## Wrap-up

## Exercises (do these next)

1. Add dropout and train on a 10× larger corpus (e.g. concatenated Python stdlib docstrings); compare val loss.
2. Implement KV-caching for `generate` and confirm identical outputs (preview of Module 23).
3. Switch pre-LN to post-LN and compare gradient norms at initialisation.

## Where to read more

- Wiki chapter: [Module 14](https://github.com/anuxlab/deep-learning-systems/wiki/Module-14) (derivations, worked example, Q&A).
- Vaswani et al. (2017) *Attention Is All You Need*; Xiong et al. (2020) pre-LN; Karpathy's *nanoGPT*; Phuong & Hutter (2022) *Formal algorithms for Transformers*.